In [1]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Subset

import sys
sys.path.insert(0, "../../..")

from utils.scaling import fit_scalers, apply_scaler
from utils.data_processing import dataset_df_to_token_df
from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_aeema_articulatory_features

In [2]:
df = pd.read_csv("../../data_processing/dataset_construction/datasets/processed_aeema.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[2])
token_df = dataset_df_to_token_df(df)

In [3]:
# token_df = token_df.sample(frac=1).reset_index(drop=True)
# token_df = token_df.iloc[int(0.95 * len(token_df)):]
# token_df.sort_values(by='start', inplace=True)
# token_df.sort_values(by='ac_f', inplace=True)

In [4]:
with open(f"../../encoder_training/scalers/aeema_art.pkl", "rb") as f:
    art_scalers = pickle.load(f)
with open(f"../../encoder_training/scalers/aeema_ac.pkl", "rb") as f:
    ac_scalers = pickle.load(f)

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from utils.models import MDN
import copy

from utils.models import SequenceAutoencoder

device = "mps" if torch.backends.mps.is_available() else "cpu"

art_model = SequenceAutoencoder(
    feature_shape=(6, 3),
    latent_dim=64,
).to(device)
ckpt = torch.load(
    "../../encoder_training/ckpts/aeema/art_model_ckpt.pth",
    map_location=device,
)
art_model.load_state_dict(ckpt["model"])
art_model.eval()

ac_model = SequenceAutoencoder(
    feature_shape=13,
    latent_dim=64,
).to(device)
ac_model.load_state_dict(torch.load("../../encoder_training/ckpts/aeema/ac_model_ckpt.pth", map_location=device))
ac_model.eval()

SequenceAutoencoder(
  (encoder_cnn): TimeMargin(
    (body): Sequential(
      (0): Conv1d(13, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (1): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (2): ReLU()
      (3): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (4): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (5): ReLU()
      (6): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (7): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (8): ReLU()
    )
  )
  (encoder_proj): Linear(in_features=128, out_features=256, bias=True)
  (trunk): TemporalTrunk(
    (encoder_gru): GRU(256, 256, batch_first=True, bidirectional=True)
    (pool): MaskedPool(
      (score): Sequential(
        (0): Linear(in_features=512, out_features=256, bias=True)
        (1): Tanh()
        (2): Linear(in_features=256, out_features=1, bias=True)
      )
    )
   

In [6]:
art_vectors = []
ac_vectors = []
phonemes = []
subjects = []
groups = []

pbar = tqdm.tqdm(token_df.iterrows(), total=len(token_df))

batch = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]

    articulatory_features = get_aeema_articulatory_features(art_f, start, end)
    if articulatory_features is None: continue
    articulatory_features = articulatory_features.reshape(articulatory_features.shape[0], 6, 3)
    articulatory_features = apply_scaler(articulatory_features, art_scalers[subject])
    articulatory_features = torch.from_numpy(articulatory_features).float().unsqueeze(0).to(device)
    art_out, art_z = art_model(articulatory_features, return_latent=True)

    mfccs = get_mfccs(ac_f, start, end)
    if mfccs is None: continue
    mfccs = apply_scaler(mfccs, ac_scalers[subject])
    mfccs = torch.from_numpy(mfccs).unsqueeze(0).to(device)
    ac_out, ac_z = ac_model(mfccs, return_latent=True)

    art_vectors.append(art_z.detach().cpu().numpy().squeeze())
    ac_vectors.append(ac_z.detach().cpu().numpy().squeeze())
    phonemes.append(phone)
    subjects.append(subject)
    groups.append(art_f)

100%|██████████| 56882/56882 [1:08:52<00:00, 13.77it/s]


In [7]:
import numpy as np
art = np.stack(art_vectors, axis=0)   # (N, T_A, D_A)
ac = np.stack(ac_vectors, axis=0)  # (N, T_X, D_X)

# A: (N, T_A, D_A)
# X: (N, T_X, D_X)
N, D = art.shape
N2, D_X = ac.shape
assert N == N2

art_flat = art.reshape(N, D)
ac_flat = ac.reshape(N, D_X)   # shape (N, x_dim)

print("art_flat:", art_flat.shape, "ac_flat:", ac_flat.shape)

art_flat: (56810, 64) ac_flat: (56810, 64)


In [8]:
art_raw = art_flat.astype(np.float32)
ac_raw = ac_flat.astype(np.float32)

In [9]:
from utils import mi

res = mi.compute_all(art_raw, ac_raw, phonemes, groups)
df  = mi.per_phoneme(res)

N=56810  art=64d  ac=64d  phones=39  device=mps

=== art|ac,phone ===
  [art|ac,phone] fold 0: val NLL    67.02   sigmas at floor  0.0%
  [art|ac,phone] fold 1: val NLL    67.72   sigmas at floor  0.0%
  [art|ac,phone] fold 2: val NLL    66.94   sigmas at floor  0.0%
  [art|ac,phone] fold 3: val NLL    65.89   sigmas at floor  0.0%
  [art|ac,phone] fold 4: val NLL    68.13   sigmas at floor  0.0%

=== art|phone ===
  [art|phone] fold 0: val NLL    79.49   sigmas at floor  0.0%
  [art|phone] fold 1: val NLL    79.10   sigmas at floor  0.0%
  [art|phone] fold 2: val NLL    79.29   sigmas at floor  0.0%
  [art|phone] fold 3: val NLL    76.35   sigmas at floor  0.0%
  [art|phone] fold 4: val NLL    79.79   sigmas at floor  0.0%

=== art|ac ===
  [art|ac] fold 0: val NLL    68.85   sigmas at floor  0.0%
  [art|ac] fold 1: val NLL    70.10   sigmas at floor  0.0%
  [art|ac] fold 2: val NLL    68.50   sigmas at floor  0.0%
  [art|ac] fold 3: val NLL    67.97   sigmas at floor  0.0%
  [art|ac]

In [10]:
df

,phone,n,n_all,kept,mi,flexibility,ci_lo,ci_hi,half_a,half_b,model_err,confusability,H_art_given_ac_phone,H_art_given_ac
0,ð,1122,1349,0.831727,5.477995,-5.477995,4.638744,6.247717,5.209922,5.746067,8.968353,6.518714,53.601035,60.119749
1,dʒ,515,593,0.868465,6.117573,-6.117573,4.843852,7.508900,6.712654,5.524798,12.337085,2.640218,56.886706,59.526924
2,f,928,1076,0.862454,7.263771,-7.263771,6.465137,8.074515,6.572649,7.954893,9.164708,3.957180,58.140741,62.097921
3,θ,268,303,0.884488,7.303355,-7.303355,5.682648,8.972331,6.163854,8.442855,9.316957,7.024827,61.843862,68.868689
4,b,1048,1253,0.836393,7.366137,-7.366137,6.604204,8.199846,7.346342,7.385931,9.163803,3.493041,56.047741,59.540782
5,h,600,700,0.857143,7.541714,-7.541714,6.357578,8.733067,8.481333,6.602095,9.802204,3.216489,56.791682,60.008172
6,p,1319,1517,0.869479,7.992840,-7.992840,7.205658,8.754932,7.868334,8.117158,10.003587,3.879186,58.739233,62.618419
7,v,789,931,0.847476,8.020143,-8.020143,7.113046,8.957466,7.655141,8.384220,8.396355,3.628885,55.570730,59.199615
8,ŋ,477,576,0.828125,8.407831,-8.407831,7.138236,9.738440,9.133076,7.685621,10.348034,4.806675,55.474674,60.281349
9,m,1385,1701,0.814227,8.469125,-8.469125,7.737592,9.145412,8.329360,8.608689,9.603389,3.729798,56.865299,60.595096


In [13]:
df.to_csv("res/aeema.csv", index=False)